# BERT 양방향 사전학습 실습 - Masked Language Modeling Data Collator

- Tutorial ID: `mod-bert-mlm-lab`
- Tutorial: BERT 양방향 사전학습 실습
- Section ID: `bert-1`
- Section: Masked Language Modeling Data Collator


# Integrated Notebook: 05_BERT_MLM_Data_Collator

## Original Version
Source: 067_mod-bert-mlm-lab_bert-1_BERT_#Uc591#Ubc29#Ud5a5_#Uc0ac#Uc804#Ud559#Uc2b5_#Uc2e4#Uc2b5_-_Masked_Language_Modeling_Data_Collator.ipynb

# BERT 양방향 사전학습 실습 - Masked Language Modeling Data Collator

- Tutorial ID: `mod-bert-mlm-lab`
- Tutorial: BERT 양방향 사전학습 실습
- Section: Masked Language Modeling Data Collator

In [ ]:
# ============================================================
# 코드 읽는 법 — Masked Language Modeling Data Collator
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#   3) 위치 정보가 벡터 회전/편향으로 attention score에 들어가는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
np.random.seed(0)
vocab = ['[PAD]','[MASK]','the','cat','sat','on','mat','dog','ran']
tok2id = {t:i for i,t in enumerate(vocab)}
seq = np.array([tok2id[t] for t in ['the','cat','sat','on','the','mat']])
mask_prob = 0.30
labels = np.full_like(seq, -100)
masked = seq.copy()
for i,t in enumerate(seq):
    if np.random.rand() < mask_prob:
        labels[i] = t
                r = np.random.rand()
                if r < .8: masked[i] = tok2id['[MASK]']
                elif r < .9: masked[i] = np.random.randint(2, len(vocab))
        else: pass
print('input :', [vocab[i] for i in masked])
print('label :', ['-' if x==-100 else vocab[x] for x in labels])
print('loss positions:', np.where(labels!=-100)[0].tolist())

## AI Developed Version 1
Source: 067_BERT_Masked_Language_Modeling_Data_Collator.ipynb

# 📘 BERT - Masked Language Modeling Data Collator

## BERT 양방향 사전학습 실습

---

### 🎯 학습 목표

1. BERT가 무엇이고 GPT와 어떻게 다른지 이해하기
2. Masked Language Modeling (MLM)의 원리 이해하기
3. Data Collator의 역할과 구현 방법 배우기
4. HuggingFace의 DataCollatorForLanguageModeling 사용해보기

### 📋 선수 지식

- 노트북 063~066의 Attention 메커니즘
- 기본적인 NLP 개념 (토큰화, 임베딩)

---

### 📖 배경: BERT란?

**BERT** = **B**idirectional **E**ncoder **R**epresentations from **T**ransformers

#### GPT vs BERT 비교

| | GPT | BERT |
|--|-----|------|
| **방향** | 단방향 (왼쪽→오른쪽) | 양방향 (좌우 모두) |
| **Mask** | Causal Mask 사용 | Mask 없음 |
| **학습 방법** | 다음 단어 예측 | 가려진 단어 예측 (MLM) |
| **비유** | 소설 쓰기 (앞에서부터) | 빈칸 채우기 시험 |
| **구조** | Transformer Decoder | Transformer Encoder |

#### Masked Language Modeling (MLM) 이란?

BERT의 핵심 학습 방법입니다.

```
원본: "나는 오늘 학교에 갔다"
마스킹: "나는 [MASK] 학교에 갔다"
BERT의 목표: [MASK] = "오늘" 을 맞추기
```

[MASK]의 양쪽 문맥("나는"과 "학교에 갔다")을 모두 볼 수 있으므로
**양방향(Bidirectional)** 학습이 가능합니다!

In [ ]:
# ============================================================
# 필요한 라이브러리 설치 및 임포트

In [ ]:
# transformers 라이브러리가 없다면 설치
# !pip install transformers datasets

import torch
import torch.nn as nn
import numpy as np
from copy import deepcopy

print(f"PyTorch 버전: {torch.__version__}")

try:
    from transformers import (
        AutoTokenizer,
        DataCollatorForLanguageModeling
    )
    print("✅ transformers 라이브러리 로드 완료")
except ImportError:
    print("❌ transformers가 필요합니다: !pip install transformers")

## 1단계: 토크나이저 이해하기

### 토크나이저란?

텍스트를 모델이 이해할 수 있는 **숫자(토큰 ID)**로 변환하는 도구입니다.

```
텍스트: "나는 학생입니다"
토큰화: ["나", "##는", "학생", "##입니다"]
토큰 ID: [2039, 2003, 9521, 18879]
```

특수 토큰:
- `[CLS]`: 문장의 시작 (분류 태스크에 사용)
- `[SEP]`: 문장의 끝 / 문장 구분
- `[MASK]`: 마스킹된 위치
- `[PAD]`: 패딩 (길이 맞추기용)

In [ ]:
# ============================================================
# 토크나이저 로드 및 기본 사용법

In [ ]:
# BERT 기본 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 기본 정보 확인
print("토크나이저 정보:")
print(f"  어휘 크기: {tokenizer.vocab_size:,}개")
print(f"  [CLS] 토큰 ID: {tokenizer.cls_token_id}")
print(f"  [SEP] 토큰 ID: {tokenizer.sep_token_id}")
print(f"  [MASK] 토큰 ID: {tokenizer.mask_token_id}")
print(f"  [PAD] 토큰 ID: {tokenizer.pad_token_id}")
print()

# 토큰화 예시
text = "I love natural language processing"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print(f"원본 텍스트: '{text}'")
print(f"토큰화 결과: {tokens}")
print(f"토큰 ID:    {token_ids}")
print(f"  (맨 앞 {token_ids[0]} = [CLS], 맨 뒤 {token_ids[-1]} = [SEP])")
print()

# 디코딩 (ID → 텍스트)
decoded = tokenizer.decode(token_ids)
print(f"디코딩 결과: '{decoded}'")

## 2단계: MLM 마스킹 규칙 이해하기

### BERT의 마스킹 전략

BERT는 입력 토큰의 **15%**를 선택하여 마스킹합니다.
선택된 토큰은 다음 비율로 처리됩니다:

| 비율 | 처리 방법 | 이유 |
|------|----------|------|
| **80%** | [MASK]로 교체 | 기본 마스킹 |
| **10%** | 랜덤 토큰으로 교체 | 모델이 모든 위치를 주의하도록 |
| **10%** | 원래 토큰 유지 | 실제 추론 시 [MASK]가 없으므로 |

왜 100% [MASK]로 바꾸지 않나요?
- 실제 사용(fine-tuning) 시에는 [MASK] 토큰이 없습니다
- 학습과 실제 사용 간의 gap을 줄이기 위해
- 모든 위치에서 올바른 표현을 학습하도록 유도

In [ ]:
# ============================================================
# MLM 마스킹을 수동으로 구현해보기

In [ ]:
def manual_mlm_masking(input_ids, tokenizer, mlm_probability=0.15):
    """
    BERT 스타일의 MLM 마스킹을 수동으로 구현합니다.
    
    Args:
        input_ids: 토큰 ID 리스트
        tokenizer: 토크나이저
        mlm_probability: 마스킹 확률 (기본 15%)
    
    Returns:
        masked_ids: 마스킹된 토큰 ID
        labels: 정답 레이블 (-100 = 예측하지 않는 위치)
    """
    # 원본 복사 (원본 변경 방지)
    masked_ids = input_ids.copy()
    labels = [-100] * len(input_ids)  # -100: PyTorch에서 loss 계산 시 무시
    
    # 특수 토큰 위치 파악 ([CLS], [SEP], [PAD] 등)
    special_token_ids = {tokenizer.cls_token_id, tokenizer.sep_token_id, 
                         tokenizer.pad_token_id}
    
    for i in range(len(input_ids)):
        # 특수 토큰은 마스킹하지 않음
        if input_ids[i] in special_token_ids:
            continue
        
        # 15% 확률로 마스킹 대상 선택
        if np.random.random() < mlm_probability:
            # 정답 레이블 설정
            labels[i] = input_ids[i]
            
            rand = np.random.random()
            
            if rand < 0.8:
                # 80% → [MASK]로 교체
                masked_ids[i] = tokenizer.mask_token_id
            elif rand < 0.9:
                # 10% → 랜덤 토큰으로 교체
                masked_ids[i] = np.random.randint(0, tokenizer.vocab_size)
            # else: 10% → 원래 토큰 유지 (아무것도 안 함)
    
    return masked_ids, labels


# 테스트
np.random.seed(42)

text = "The cat sat on the mat and looked at the dog"
input_ids = tokenizer.encode(text)
tokens = tokenizer.convert_ids_to_tokens(input_ids)

print(f"원본 텍스트: '{text}'")
print(f"토큰:       {tokens}")
print(f"토큰 ID:    {input_ids}")
print()

# 마스킹 적용 (여러 번 해보면 다른 결과가 나옵니다)
for trial in range(3):
    masked_ids, labels = manual_mlm_masking(input_ids, tokenizer)
    masked_tokens = tokenizer.convert_ids_to_tokens(masked_ids)
    
    print(f"\n시도 {trial + 1}:")
    print(f"  마스킹된 토큰: {masked_tokens}")
    
    # 마스킹된 위치와 정답 표시
    for i, (orig, masked, label) in enumerate(zip(tokens, masked_tokens, labels)):
        if label != -100:
            label_token = tokenizer.convert_ids_to_tokens([label])[0]
            print(f"    위치 {i}: '{orig}' → '{masked}' (정답: '{label_token}')")

## 3단계: Data Collator 이해하기

### Data Collator란?

**Data Collator**는 개별 데이터 샘플들을 **배치(batch)**로 묶어주는 도구입니다.

MLM용 Data Collator의 역할:
1. 여러 문장을 같은 길이로 패딩
2. 각 문장에 랜덤 마스킹 적용
3. 정답 레이블 생성
4. 텐서로 변환

```python
# Data Collator가 하는 일
개별 샘플들 → [패딩 + 마스킹 + 레이블 생성] → 배치 텐서
```

In [ ]:
# ============================================================
# 수동 Data Collator 구현

In [ ]:
class SimpleMLMDataCollator:
    """
    간단한 MLM Data Collator.
    
    개별 토큰화된 샘플들을 받아서:
    1. 같은 길이로 패딩
    2. 마스킹 적용
    3. 배치 텐서로 변환
    """
    
    def __init__(self, tokenizer, mlm_probability=0.15):
        self.tokenizer = tokenizer
        self.mlm_probability = mlm_probability
    
    def __call__(self, examples):
        """
        Args:
            examples: 토큰화된 샘플 리스트
                      각 샘플: {'input_ids': [101, 2023, ...], 'attention_mask': [1, 1, ...]}
        
        Returns:
            배치 딕셔너리 (텐서)
        """
        # Step 1: 최대 길이 찾기
        max_len = max(len(e['input_ids']) for e in examples)
        
        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []
        
        for example in examples:
            ids = example['input_ids']
            
            # Step 2: 패딩
            padding_length = max_len - len(ids)
            padded_ids = ids + [self.tokenizer.pad_token_id] * padding_length
            attention_mask = [1] * len(ids) + [0] * padding_length
            
            # Step 3: 마스킹
            masked_ids, labels = manual_mlm_masking(
                padded_ids, self.tokenizer, self.mlm_probability
            )
            
            batch_input_ids.append(masked_ids)
            batch_attention_mask.append(attention_mask)
            batch_labels.append(labels)
        
        # Step 4: 텐서로 변환
        return {
            'input_ids': torch.tensor(batch_input_ids),
            'attention_mask': torch.tensor(batch_attention_mask),
            'labels': torch.tensor(batch_labels),
        }


# 테스트
collator = SimpleMLMDataCollator(tokenizer)

# 여러 문장 토큰화
sentences = [
    "The cat sat on the mat",
    "I love programming",
    "Natural language processing is fun and exciting",
]

# 토큰화
tokenized = [tokenizer(s, add_special_tokens=True) for s in sentences]

print("각 문장의 길이:")
for i, s in enumerate(sentences):
    print(f"  '{s}' → {len(tokenized[i]['input_ids'])} 토큰")
print()

# Data Collator 적용
batch = collator(tokenized)

print("배치 결과:")
print(f"  input_ids shape:      {batch['input_ids'].shape}")
print(f"  attention_mask shape: {batch['attention_mask'].shape}")
print(f"  labels shape:         {batch['labels'].shape}")
print()

# 마스킹 결과 확인
for i in range(len(sentences)):
    ids = batch['input_ids'][i].tolist()
    labels = batch['labels'][i].tolist()
    decoded = tokenizer.decode(ids)
    print(f"문장 {i+1}: {decoded}")
    
    masked_positions = [j for j, l in enumerate(labels) if l != -100]
    if masked_positions:
        for pos in masked_positions:
            original = tokenizer.decode([labels[pos]])
            current = tokenizer.decode([ids[pos]])
            print(f"  위치 {pos}: '{current}' → 정답: '{original}'")

## 4단계: HuggingFace DataCollatorForLanguageModeling

실전에서는 HuggingFace의 내장 Data Collator를 사용합니다.
우리가 구현한 것과 동일한 기능을 제공합니다.

In [ ]:
# ============================================================
# HuggingFace DataCollatorForLanguageModeling 사용

In [ ]:
# HuggingFace 내장 Collator 생성
hf_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,              # MLM 모드 활성화
    mlm_probability=0.15,  # 15% 마스킹
    return_tensors="pt"    # PyTorch 텐서 반환
)

# 동일한 문장들로 테스트
tokenized_hf = [
    tokenizer(s, return_tensors=None, add_special_tokens=True)
    for s in sentences
]

# Collator 적용
batch_hf = hf_collator(tokenized_hf)

print("HuggingFace DataCollator 결과:")
print(f"  input_ids shape:      {batch_hf['input_ids'].shape}")
print(f"  attention_mask shape: {batch_hf['attention_mask'].shape}")
print(f"  labels shape:         {batch_hf['labels'].shape}")
print()

# 결과 확인
for i in range(len(sentences)):
    ids = batch_hf['input_ids'][i].tolist()
    labels = batch_hf['labels'][i].tolist()
    print(f"문장 {i+1}: {tokenizer.decode(ids)}")
    
    masked = [(j, tokenizer.decode([ids[j]]), tokenizer.decode([labels[j]])) 
              for j, l in enumerate(labels) if l != -100]
    for pos, current, original in masked:
        print(f"  위치 {pos}: '{current}' → 정답: '{original}'")
    if not masked:
        print(f"  (이 문장에서는 마스킹된 토큰 없음)")

## 5단계: Labels의 -100 이해하기

### -100의 의미

PyTorch의 `CrossEntropyLoss`에서 `ignore_index=-100`이 기본값입니다.
즉, 레이블이 -100인 위치는 **loss 계산에서 제외**됩니다.

```
입력:  [CLS] I [MASK] cats [SEP]
레이블: -100 -100 love -100 -100

→ [MASK] 위치만 loss를 계산!
→ 다른 위치는 무시 (예측할 필요 없음)
```

In [ ]:
# ============================================================
# -100과 CrossEntropyLoss 동작 확인

In [ ]:
# 간단한 예시
# 어휘 크기 = 5인 작은 모델 가정
vocab_size = 5
seq_len = 4

# 모델의 예측 (logits): (batch=1, seq=4, vocab=5)
logits = torch.randn(1, seq_len, vocab_size)

# 레이블: 2번째 위치만 예측 (나머지 -100)
labels = torch.tensor([[-100, 3, -100, -100]])  # 2번째 위치의 정답 = 토큰 ID 3

print("모델 예측 (logits):")
print(logits[0])
print()
print("레이블:", labels[0].tolist())
print("  → -100인 위치는 loss에서 제외됩니다")
print()

# Loss 계산
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

# logits를 (batch*seq, vocab)으로, labels를 (batch*seq,)로 reshape
loss = loss_fn(
    logits.view(-1, vocab_size),  # (4, 5)
    labels.view(-1)               # (4,)
)

print(f"Loss: {loss.item():.4f}")
print("  → 2번째 위치(정답=3)의 예측에 대해서만 loss가 계산됩니다")
print()

# 비교: 수동 계산
manual_loss = nn.CrossEntropyLoss()(logits[0, 1:2], labels[0, 1:2])
print(f"수동 계산 Loss: {manual_loss.item():.4f}")
print(f"동일한 결과: {abs(loss.item() - manual_loss.item()) < 1e-6}")

## 🔑 핵심 정리

### MLM Data Collator의 역할

```
텍스트 데이터
  │
  ▼ 토크나이저
토큰 ID 시퀀스들 (길이가 다름)
  │
  ▼ Data Collator
  ├─ 1. 패딩 (같은 길이로)
  ├─ 2. 마스킹 (15% 토큰)
  │     ├─ 80% → [MASK]
  │     ├─ 10% → 랜덤 토큰
  │     └─ 10% → 원래 유지
  ├─ 3. 레이블 생성 (-100 / 원래 토큰 ID)
  └─ 4. 텐서 변환
  │
  ▼
배치 텐서 → BERT 모델에 입력
```

### 중요 개념

| 개념 | 설명 |
|------|------|
| **MLM** | 15% 토큰을 가리고 맞추는 학습 |
| **Data Collator** | 샘플들을 배치로 묶고 마스킹하는 도구 |
| **-100** | Loss 계산에서 제외되는 위치 표시 |
| **80/10/10 규칙** | [MASK]/랜덤/유지 비율 |

### 다음 단계

- **노트북 068**: MLM 마스킹 상세 구현

## AI Developed Version 2
Source: 05_BERT_MLM_Data_Collator.ipynb

# 📚 Notebook 5: BERT의 Masked Language Modeling (MLM) 구현

## 양방향 사전학습의 핵심 - Data Collator 구현

---

### 🎯 학습 목표
BERT의 사전학습 방법인 **Masked Language Modeling (MLM)**을 이해하고,
학습 데이터를 준비하는 **Data Collator**를 구현합니다.

### 💡 MLM이란?

BERT는 문장의 일부 단어를 **[MASK]**로 가리고, 원래 단어를 맞추도록 학습합니다.

```
원본: "나는 고양이를 좋아한다"
마스킹: "나는 [MASK]를 좋아한다"
목표: [MASK] → "고양이" (원래 단어 맞추기)
```

### 왜 MLM을 사용할까?

**GPT (Causal LM):**
- 왼쪽에서 오른쪽으로만 봄 (단방향)
- "나는 고양이를" → 다음 단어 예측

**BERT (Masked LM):**
- 양쪽 모두 봄 (양방향)
- "나는 [MASK]를 좋아한다" → 빈칸 채우기
- 앞뒤 문맥을 모두 활용하므로 더 풍부한 표현 학습!

### 📋 목차
1. MLM의 마스킹 전략 이해
2. 토크나이저 기초
3. 마스킹 함수 구현 (Step by Step)
4. Data Collator 구현
5. 실제 데이터로 테스트

---

In [ ]:
# ============================================================
# 환경 설정

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import copy
import numpy as np

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

print("✅ 라이브러리 로딩 완료!")

## 1. BERT의 마스킹 전략 이해 🎭

BERT 논문에서 제안한 마스킹 전략:

전체 토큰의 **15%**를 마스킹 대상으로 선정하고, 그 중:

| 비율 | 처리 방법 | 이유 |
|------|-----------|------|
| **80%** | `[MASK]` 토큰으로 교체 | 모델이 빈칸을 채우도록 학습 |
| **10%** | 랜덤한 다른 토큰으로 교체 | 모델이 모든 위치를 주시하도록 |
| **10%** | 원래 토큰 유지 | 실제 사용 시 [MASK]가 없으므로 대비 |

### 왜 이렇게 복잡하게 할까?

**80% [MASK]만 사용하면 생기는 문제:**
- 학습할 때는 [MASK]가 있지만, 실제 사용(fine-tuning)할 때는 [MASK]가 없음
- 이 불일치(mismatch)를 줄이기 위해 10%+10% 전략 사용

**10% 랜덤 교체의 이유:**
- 모든 위치에서 올바른 토큰이 있는지 확인하도록 강제
- 모델이 게으르게 되는 것을 방지

In [ ]:
# ============================================================
# 마스킹 전략 시각화

In [ ]:
print("📊 BERT 마스킹 전략 시뮬레이션")
print("=" * 60)

# 예시 문장
original = ['나는', '어제', '도서관', '에서', '책', '을', '읽었다']
print(f"원본 문장: {' '.join(original)}")
print(f"토큰 수: {len(original)}")
print(f"마스킹 대상 수 (15%): {max(1, int(len(original) * 0.15))}개\n")

# 15%에 해당하는 토큰 선정 (최소 1개)
num_to_mask = max(1, int(len(original) * 0.15))
mask_indices = random.sample(range(len(original)), num_to_mask)
print(f"마스킹 대상 인덱스: {mask_indices}")
print(f"마스킹 대상 토큰: {[original[i] for i in mask_indices]}\n")

# 각 마스킹 전략 시뮬레이션 (10번)
vocab = ['고양이', '강아지', '학교', '집', '먹다', '자다', '가다']

for trial in range(5):
    masked = original.copy()
    strategy = []
    for idx in mask_indices:
        r = random.random()  # 0~1 사이 랜덤 값
        if r < 0.8:
            masked[idx] = '[MASK]'
            strategy.append(f"[MASK] (80%)")
        elif r < 0.9:
            masked[idx] = random.choice(vocab)
            strategy.append(f"랜덤→'{masked[idx]}' (10%)")
        else:
            strategy.append(f"유지 (10%)")
    
    print(f"시도 {trial+1}: {' '.join(masked)}")
    print(f"         전략: {', '.join(strategy)}")

## 2. 간단한 토크나이저 만들기

실제로는 HuggingFace의 토크나이저를 사용하지만,
이해를 위해 간단한 토크나이저를 직접 만들어 봅시다.

**토크나이저(Tokenizer)란?**
- 텍스트를 숫자(ID)로 변환하는 도구
- 각 단어/토큰에 고유한 번호를 부여
- 모델은 숫자만 이해하므로 필수적인 과정

In [ ]:
# ============================================================
# 간단한 토크나이저 구현

In [ ]:
class SimpleTokenizer:
    """
    BERT 스타일의 간단한 토크나이저
    
    특수 토큰:
    - [PAD]: 패딩 (길이 맞추기용, ID=0)
    - [UNK]: 모르는 단어 (ID=1)
    - [CLS]: 문장 시작 (분류용, ID=2)
    - [SEP]: 문장 구분 (ID=3)
    - [MASK]: 마스킹 (MLM용, ID=4)
    """
    
    def __init__(self):
        # 특수 토큰 정의
        self.special_tokens = {
            '[PAD]': 0,
            '[UNK]': 1,
            '[CLS]': 2,
            '[SEP]': 3,
            '[MASK]': 4,
        }
        
        # 일반 단어 사전
        words = [
            '나는', '너는', '우리', '그는', '그녀는',
            '고양이', '강아지', '새', '물고기', '토끼',
            '를', '을', '이', '가', '에서', '에', '와', '과',
            '좋아한다', '싫어한다', '키운다', '먹는다', '본다',
            '집', '학교', '공원', '도서관', '병원',
            '어제', '오늘', '내일', '항상', '가끔',
            '크다', '작다', '예쁘다', '귀엽다', '빠르다',
            '읽다', '쓰다', '달리다', '걷다', '잠을잔다'
        ]
        
        # 전체 어휘 사전 구성
        self.token2id = {**self.special_tokens}
        for i, word in enumerate(words):
            self.token2id[word] = len(self.special_tokens) + i
        
        # 역방향 매핑 (ID → 토큰)
        self.id2token = {v: k for k, v in self.token2id.items()}
        
        self.vocab_size = len(self.token2id)
        self.pad_token_id = self.special_tokens['[PAD]']
        self.mask_token_id = self.special_tokens['[MASK]']
        self.cls_token_id = self.special_tokens['[CLS]']
        self.sep_token_id = self.special_tokens['[SEP]']
    
    def encode(self, text):
        """텍스트를 토큰 ID 리스트로 변환"""
        tokens = text.split()  # 공백 기준 분리 (간단한 구현)
        # [CLS] + 토큰들 + [SEP]
        ids = [self.cls_token_id]
        for token in tokens:
            ids.append(self.token2id.get(token, self.special_tokens['[UNK]']))
        ids.append(self.sep_token_id)
        return ids
    
    def decode(self, ids):
        """토큰 ID 리스트를 텍스트로 변환"""
        return ' '.join([self.id2token.get(id, '[UNK]') for id in ids])

# 토크나이저 생성 및 테스트
tokenizer = SimpleTokenizer()

print(f"어휘 크기: {tokenizer.vocab_size}")
print(f"\n특수 토큰:")
for token, id in tokenizer.special_tokens.items():
    print(f"  {token} → ID: {id}")

# 인코딩/디코딩 테스트
test_text = "나는 고양이 를 좋아한다"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)

print(f"\n📝 인코딩 테스트:")
print(f"  원본: '{test_text}'")
print(f"  인코딩: {encoded}")
print(f"  디코딩: '{decoded}'")

## 3. 마스킹 함수 구현 (Step by Step)

이제 BERT의 마스킹 로직을 구현합니다.

In [ ]:
# ============================================================
# MLM 마스킹 함수 구현

In [ ]:
def create_mlm_data(input_ids, tokenizer, mlm_prob=0.15):
    """
    BERT의 Masked Language Modeling 데이터를 생성합니다.
    
    Args:
        input_ids (list): 토큰 ID 리스트 (예: [2, 5, 10, 18, 3])
        tokenizer: 토크나이저 객체
        mlm_prob (float): 마스킹 확률 (기본 15%)
    
    Returns:
        masked_ids (list): 마스킹된 토큰 ID 리스트
        labels (list): 정답 레이블 (-100은 예측하지 않는 위치)
    """
    # 입력을 복사 (원본 변경 방지)
    masked_ids = input_ids.copy()
    
    # 레이블 초기화: -100은 "이 위치는 예측하지 마세요"라는 의미
    # PyTorch의 CrossEntropyLoss에서 -100은 무시됨
    labels = [-100] * len(input_ids)
    
    # 특수 토큰 ID 모음 (마스킹하면 안 되는 토큰들)
    special_ids = set(tokenizer.special_tokens.values())
    
    # 마스킹 가능한 위치 찾기 (특수 토큰 제외)
    maskable_indices = [
        i for i, id in enumerate(input_ids) 
        if id not in special_ids
    ]
    
    # 마스킹할 토큰 수 계산
    num_to_mask = max(1, int(len(maskable_indices) * mlm_prob))
    
    # 랜덤으로 마스킹 위치 선택
    mask_indices = random.sample(maskable_indices, num_to_mask)
    
    for idx in mask_indices:
        # 정답 레이블 설정 (원래 토큰 ID)
        labels[idx] = input_ids[idx]
        
        r = random.random()
        
        if r < 0.8:
            # 80%: [MASK]로 교체
            masked_ids[idx] = tokenizer.mask_token_id
        elif r < 0.9:
            # 10%: 랜덤 토큰으로 교체
            # 특수 토큰을 제외한 랜덤 토큰 선택
            random_id = random.randint(
                len(tokenizer.special_tokens),  # 특수 토큰 이후부터
                tokenizer.vocab_size - 1         # 마지막 토큰까지
            )
            masked_ids[idx] = random_id
        # else: 10%는 원래 토큰 유지 (아무것도 안 함)
    
    return masked_ids, labels

print("✅ create_mlm_data 함수 구현 완료!")

In [ ]:
# ============================================================
# 마스킹 함수 테스트

In [ ]:
# 테스트 문장
test_sentence = "나는 고양이 를 좋아한다"
input_ids = tokenizer.encode(test_sentence)

print("📝 마스킹 테스트")
print("=" * 60)
print(f"원본 문장: {test_sentence}")
print(f"토큰 ID:  {input_ids}")
print(f"디코딩:   {tokenizer.decode(input_ids)}")

# 여러 번 마스킹 수행 (다른 결과 관찰)
print(f"\n--- 마스킹 결과 (5번 수행) ---\n")
for trial in range(5):
    masked_ids, labels = create_mlm_data(input_ids, tokenizer)
    
    masked_text = tokenizer.decode(masked_ids)
    label_tokens = []
    for i, label in enumerate(labels):
        if label != -100:
            label_tokens.append(f"위치{i}: {tokenizer.id2token[label]}")
    
    print(f"  시도 {trial+1}:")
    print(f"    마스킹된 문장: {masked_text}")
    print(f"    정답 레이블: {label_tokens}")
    print(f"    labels 배열: {labels}")
    print()

## 4. Data Collator 구현

**Data Collator란?**
- 여러 개의 샘플을 하나의 **배치(batch)**로 묶어주는 역할
- 각 샘플의 길이가 다를 수 있으므로 **패딩(padding)**도 수행
- MLM의 마스킹도 여기서 동적으로 수행 (매 배치마다 다른 마스킹!)

In [ ]:
# ============================================================
# MLM Data Collator 구현

In [ ]:
class MLMDataCollator:
    """
    BERT MLM을 위한 Data Collator
    
    역할:
    1. 여러 샘플을 배치로 묶기
    2. 패딩 추가 (길이 맞추기)
    3. MLM 마스킹 적용
    4. Attention Mask 생성
    """
    
    def __init__(self, tokenizer, mlm_prob=0.15, max_length=32):
        """
        Args:
            tokenizer: 토크나이저
            mlm_prob: 마스킹 확률 (기본 15%)
            max_length: 최대 시퀀스 길이
        """
        self.tokenizer = tokenizer
        self.mlm_prob = mlm_prob
        self.max_length = max_length
    
    def __call__(self, texts):
        """
        텍스트 리스트를 배치 텐서로 변환합니다.
        
        Args:
            texts (list of str): 텍스트 리스트
        
        Returns:
            dict: {
                'input_ids': 마스킹된 입력 텐서,
                'attention_mask': 어텐션 마스크 텐서,
                'labels': 정답 레이블 텐서
            }
        """
        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []
        
        # Step 1: 각 텍스트를 인코딩
        all_input_ids = [self.tokenizer.encode(text) for text in texts]
        
        # Step 2: 최대 길이 결정 (배치 내 최대 or max_length 중 작은 것)
        max_len = min(
            max(len(ids) for ids in all_input_ids),
            self.max_length
        )
        
        for input_ids in all_input_ids:
            # 길이 제한
            input_ids = input_ids[:max_len]
            
            # Step 3: MLM 마스킹 적용
            masked_ids, labels = create_mlm_data(
                input_ids, self.tokenizer, self.mlm_prob
            )
            
            # Step 4: 패딩 추가
            padding_length = max_len - len(masked_ids)
            
            # Attention Mask: 1 = 실제 토큰, 0 = PAD
            attention_mask = [1] * len(masked_ids) + [0] * padding_length
            
            # PAD 토큰 추가
            masked_ids = masked_ids + [self.tokenizer.pad_token_id] * padding_length
            labels = labels + [-100] * padding_length  # PAD 위치는 예측 안 함
            
            batch_input_ids.append(masked_ids)
            batch_attention_mask.append(attention_mask)
            batch_labels.append(labels)
        
        # Step 5: 텐서로 변환
        return {
            'input_ids': torch.tensor(batch_input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(batch_attention_mask, dtype=torch.long),
            'labels': torch.tensor(batch_labels, dtype=torch.long),
        }

print("✅ MLMDataCollator 구현 완료!")

In [ ]:
# ============================================================
# Data Collator 테스트

In [ ]:
collator = MLMDataCollator(tokenizer, mlm_prob=0.15)

# 테스트 문장들 (길이가 다름)
texts = [
    "나는 고양이 를 좋아한다",
    "너는 강아지 를 키운다",
    "우리 가 공원 에서 걷다",
]

# Data Collator 실행
batch = collator(texts)

print("📦 배치 데이터 생성 결과")
print("=" * 60)
print(f"\ninput_ids shape: {batch['input_ids'].shape}")
print(f"attention_mask shape: {batch['attention_mask'].shape}")
print(f"labels shape: {batch['labels'].shape}")

for i, text in enumerate(texts):
    print(f"\n--- 문장 {i+1}: '{text}' ---")
    print(f"  마스킹된 입력: {tokenizer.decode(batch['input_ids'][i].tolist())}")
    print(f"  Attention Mask: {batch['attention_mask'][i].tolist()}")
    
    # 정답 위치 표시
    label_info = []
    for j, label in enumerate(batch['labels'][i].tolist()):
        if label != -100:
            label_info.append(f"위치{j}={tokenizer.id2token[label]}")
    print(f"  정답 레이블: {label_info if label_info else '(마스킹 없음)'}")

## 5. 간단한 MLM 모델로 학습 시뮬레이션 🤖

Data Collator가 제대로 동작하는지 간단한 모델로 확인해봅시다.

In [ ]:
# ============================================================
# 간단한 MLM 모델 (학습 시뮬레이션용)

In [ ]:
class SimpleBERTforMLM(nn.Module):
    """
    간단한 BERT MLM 모델 (학습 시뮬레이션용)
    
    실제 BERT 구조:
    입력 → 임베딩 → Transformer 블록 × 12 → 예측 헤드 → 단어 예측
    
    여기서는 간소화:
    입력 → 임베딩 → 선형 변환 1개 → 예측 헤드 → 단어 예측
    """
    
    def __init__(self, vocab_size, d_model=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.fc = nn.Linear(d_model, d_model)
        self.prediction_head = nn.Linear(d_model, vocab_size)
    
    def forward(self, input_ids):
        x = self.embedding(input_ids)     # (batch, seq_len, d_model)
        x = F.relu(self.fc(x))            # 간단한 변환
        logits = self.prediction_head(x)  # (batch, seq_len, vocab_size)
        return logits

# 모델 생성
model = SimpleBERTforMLM(tokenizer.vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

# 학습 시뮬레이션 (5 에폭)
print(f"\n🏋️ 학습 시뮬레이션 (5 에폭)")
print("=" * 40)

for epoch in range(5):
    # 매 에폭마다 새로운 마스킹 (동적 마스킹)
    batch = collator(texts)
    
    # Forward pass
    logits = model(batch['input_ids'])  # (batch, seq_len, vocab_size)
    
    # Loss 계산
    # CrossEntropyLoss는 labels=-100인 위치를 자동으로 무시
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
    
    # logits: (batch*seq_len, vocab_size), labels: (batch*seq_len)
    loss = loss_fn(
        logits.view(-1, tokenizer.vocab_size),
        batch['labels'].view(-1)
    )
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # 마스킹된 위치의 예측 확인
    predictions = logits.argmax(dim=-1)  # 가장 높은 확률의 토큰
    
    # 정확도 계산 (마스킹된 위치만)
    mask_positions = batch['labels'] != -100
    if mask_positions.sum() > 0:
        correct = (predictions[mask_positions] == batch['labels'][mask_positions]).sum()
        total = mask_positions.sum()
        acc = correct.item() / total.item() * 100
    else:
        acc = 0
    
    print(f"  Epoch {epoch+1}: Loss = {loss.item():.4f}, 정확도 = {acc:.1f}%")

print(f"\n💡 실제 BERT는 수백만 문장으로 수십 에폭을 학습합니다.")
print(f"   여기서는 개념 확인을 위한 간단한 시뮬레이션입니다.")

## 📝 핵심 정리

### BERT MLM의 핵심:

| 구성 요소 | 역할 |
|-----------|------|
| **토크나이저** | 텍스트 → 숫자 ID 변환 |
| **마스킹 전략** | 80% [MASK] + 10% 랜덤 + 10% 유지 |
| **Data Collator** | 마스킹 + 패딩 + 배치 생성 |
| **Labels** | -100 = 무시, 나머지 = 정답 토큰 ID |

### 다음 노트북 예고 🔜
Notebook 6에서는 **MLM 마스킹 전략을 더 깊이** 파고들며,
Whole Word Masking, Dynamic Masking 등 고급 기법을 다룹니다!

## AI Developed Version 3
Source: 067_BERT_MLM_Data_Collator.ipynb

# 📘 실습 067: BERT 양방향 사전학습 - Masked Language Modeling

## 🎯 BERT란 무엇인가?

**BERT** (Bidirectional Encoder Representations from Transformers)

### 핵심 아이디어: 빈 칸 채우기!

```
학생이 공부할 때 쓰는 방법:
  원문: "파리는 프랑스의 수도이다"
  마스킹: "파리는 [MASK]의 수도이다"
  예측: [MASK] → "프랑스"

BERT는 이걸 수백억 문장으로 학습!
→ 문장의 맥락을 깊이 이해하게 됨
```

### GPT와의 차이:
| | BERT | GPT |
|--|------|-----|
| 방향 | 양방향 (←→) | 단방향 (→) |
| 학습 방식 | 빈칸 채우기 (MLM) | 다음 단어 예측 |
| 강점 | 이해 (분류, QA) | 생성 |

## 📋 이번 실습의 주제: Data Collator

MLM 학습을 위해 **자동으로 토큰을 마스킹**하는 DataCollator를 구현합니다.

### BERT 마스킹 규칙 (논문에서):
- 전체 토큰의 **15%**를 랜덤으로 선택
- 선택된 토큰의:
  - **80%** → `[MASK]`로 교체
  - **10%** → 랜덤 토큰으로 교체 (노이즈)
  - **10%** → 원래 토큰 유지 (정답과 함께)

이 복잡한 규칙이 왜 필요한지도 설명합니다!

In [ ]:
# ============================================================
# 📦 라이브러리 설치 및 임포트

In [ ]:
# Hugging Face transformers 설치
# !pip install transformers datasets

import torch
import torch.nn as nn
import numpy as np
from transformers import (
    BertTokenizer,           # BERT 토크나이저
    DataCollatorForLanguageModeling  # MLM 데이터 콜레이터
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

print("✅ 라이브러리 로드 완료")
print("BERT 토크나이저 로드 중...")

In [ ]:
# ============================================================
# 🔤 Step 1: BERT 토크나이저 이해하기

In [ ]:
#
# BERT는 WordPiece 토크나이저를 사용합니다.
# 단어를 서브워드(sub-word)로 분리해서 처리합니다.
#
# 예: "unhappiness" → ["un", "##happiness"]
#     "running"    → ["run", "##ning"]
# (## 은 앞 단어의 이어지는 부분임을 표시)

# 사전 학습된 BERT 토크나이저 로드
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

print("=" * 60)
print("BERT 토크나이저 기본 정보")
print("=" * 60)
print(f"어휘 크기: {tokenizer.vocab_size:,}")
print(f"\n특수 토큰:")
print(f"  [PAD]  = {tokenizer.pad_token_id}  ← 패딩 (짧은 문장 채우기)")
print(f"  [UNK]  = {tokenizer.unk_token_id}  ← 미지의 단어")
print(f"  [CLS]  = {tokenizer.cls_token_id}  ← 문장 시작 (분류 태스크용)")
print(f"  [SEP]  = {tokenizer.sep_token_id}  ← 문장 구분자")
print(f"  [MASK] = {tokenizer.mask_token_id} ← MLM 마스크")
print()

# 토크나이징 예시
sample_sentence = "The quick brown fox jumps over the lazy dog."
tokens = tokenizer.tokenize(sample_sentence)
token_ids = tokenizer.encode(sample_sentence)

print(f"입력 문장: {sample_sentence}")
print(f"\n토크나이징 결과:")
print(f"  토큰: {tokens}")
print(f"  토큰 수: {len(tokens)}")
print()
print(f"인코딩 결과 (토큰 ID):")
print(f"  IDs: {token_ids}")
print(f"  설명: {token_ids[0]} = [CLS], {token_ids[-1]} = [SEP]")

In [ ]:
# ============================================================
# 🎭 Step 2: BERT 마스킹 규칙 이해하기

In [ ]:
#
# 왜 복잡한 마스킹 규칙을 쓸까?
#
# [MASK] 80% + 랜덤 10% + 원본 10%
#
# 문제: [MASK]만 사용하면...
# → 모델이 '[MASK] 토큰이 있을 때만 예측하면 된다'고 생각
# → Fine-tuning 시 [MASK]가 없으면 모델이 당황!
# → 사전학습과 실제 사용 간의 불일치(mismatch)
#
# 해결책: 일부는 랜덤 토큰, 일부는 원본 유지
# → 모델이 항상 '이 토큰이 올바른가?'를 고려하게 됨
# → 사전학습-Fine-tuning 불일치 문제 완화

print("=" * 60)
print("BERT 마스킹 규칙 상세 설명")
print("=" * 60)

fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')

# 마스킹 규칙 시각화
categories = ['전체 토큰\n(100%)', '마스킹 선택\n(15%)', 
              '[MASK]\n(80%)', '랜덤 토큰\n(10%)', '원본 유지\n(10%)']
values = [100, 15, 12, 1.5, 1.5]  # 전체의 %
colors = ['#3498db', '#e67e22', '#e74c3c', '#9b59b6', '#27ae60']

# 수평 막대 그래프
bars = ax.barh([4, 3, 2, 1, 0], [100, 15, 12, 1.5, 1.5], 
               color=colors, alpha=0.8, edgecolor='black')

labels = [
    '전체 토큰 (100%)',
    '→ 15% 랜덤 선택 (마스킹 대상)',
    '→ 선택된 것의 80% = [MASK]로 교체',
    '→ 선택된 것의 10% = 랜덤 토큰으로 교체',
    '→ 선택된 것의 10% = 원본 그대로 유지'
]

for i, (bar, label) in enumerate(zip(bars, labels)):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=10)

ax.set_xlim(0, 150)
ax.set_title('BERT MLM 마스킹 규칙\n(전체 토큰 중 15%만 마스킹, 나머지 85%는 그대로)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bert_masking_rule.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n📖 마스킹 규칙 요약:")
print("  전체 토큰 중 15%를 선택하여:")
print("    - 80% → [MASK] 토큰으로 교체 (모델이 맞혀야 함)")
print("    - 10% → 랜덤 토큰으로 교체 (항상 올바른지 체크하게 만듦)")
print("    - 10% → 원본 유지 (정답이 원본임을 기억하게 함)")
print("\n  나머지 85%는 건드리지 않음!")

In [ ]:
# ============================================================
# 🏗️ Step 3: MLM Data Collator 직접 구현

In [ ]:
#
# DataCollator = 개별 데이터 샘플을 모아서 배치로 만드는 함수
# MLM DataCollator = 배치 만들면서 자동으로 마스킹까지!

class BertMLMDataCollator:
    """
    BERT MLM(Masked Language Modeling)을 위한 Data Collator
    
    동작:
    1. 배치의 문장들을 같은 길이로 패딩
    2. BERT 마스킹 규칙에 따라 토큰 마스킹
    3. 모델 입력(masked)과 정답 레이블(original) 반환
    
    사용법:
    collator = BertMLMDataCollator(tokenizer)
    batch = collator([sample1, sample2, ...])
    """
    
    def __init__(self, tokenizer, mlm_probability=0.15):
        """
        Args:
            tokenizer:       BERT 토크나이저
            mlm_probability: 마스킹할 토큰 비율 (기본값: 0.15 = 15%)
        """
        self.tokenizer = tokenizer
        self.mlm_probability = mlm_probability
    
    def __call__(self, examples):
        """
        여러 예제를 받아 마스킹된 배치 반환
        
        Args:
            examples: 토큰 ID 리스트의 리스트
                      예: [[101, 7592, 2088, 102], [101, 2023, 102]]
        
        Returns:
            dict:
                'input_ids': 마스킹된 토큰 ID
                'labels':    원본 토큰 ID (-100 = 마스킹 안 된 위치, 무시)
                'attention_mask': 패딩 위치 표시
        """
        # ─── 1단계: 패딩으로 같은 길이 맞추기 ───
        max_length = max(len(e) for e in examples)
        
        input_ids_list = []
        attention_masks = []
        
        for example in examples:
            padding_length = max_length - len(example)
            
            # 오른쪽에 [PAD] 추가
            padded = example + [self.tokenizer.pad_token_id] * padding_length
            
            # Attention mask: 실제 토큰=1, 패딩=0
            attn_mask = [1] * len(example) + [0] * padding_length
            
            input_ids_list.append(padded)
            attention_masks.append(attn_mask)
        
        input_ids = torch.tensor(input_ids_list, dtype=torch.long)
        attention_mask = torch.tensor(attention_masks, dtype=torch.long)
        
        # ─── 2단계: MLM 마스킹 적용 ───
        input_ids, labels = self.mask_tokens(input_ids)
        
        return {
            'input_ids': input_ids,
            'labels': labels,
            'attention_mask': attention_mask
        }
    
    def mask_tokens(self, inputs):
        """
        BERT 마스킹 규칙 적용
        
        Args:
            inputs: 토큰 ID 텐서  shape = (batch, seq_len)
        
        Returns:
            masked_inputs: 마스킹된 입력  shape = (batch, seq_len)
            labels:        정답 레이블    shape = (batch, seq_len)
                          마스킹 안 된 위치는 -100 (loss 계산 제외)
        """
        labels = inputs.clone()  # 원본 복사 (정답용)
        
        # ─── 마스킹할 위치 선택 ───
        # 각 토큰을 15% 확률로 마스킹 대상으로 선택
        probability_matrix = torch.full(inputs.shape, self.mlm_probability)
        
        # 특수 토큰([CLS], [SEP], [PAD])은 마스킹하지 않음
        # → 이 토큰들은 문장 구조상 중요해서 예측 대상에서 제외
        special_tokens_mask = [
            self.tokenizer.get_special_tokens_mask(val.tolist(), already_has_special_tokens=True)
            for val in inputs
        ]
        special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)
        
        # 베르누이 분포로 마스킹 위치 결정
        # torch.bernoulli: 확률 p로 1, (1-p)로 0 반환
        masked_indices = torch.bernoulli(probability_matrix).bool()
        
        # 마스킹 안 된 위치는 labels=-100 (loss 계산에서 제외)
        # 왜 -100? PyTorch CrossEntropyLoss의 ignore_index 기본값
        labels[~masked_indices] = -100
        
        # ─── 3가지 교체 전략 ───
        masked_inputs = inputs.clone()
        
        # 전략 1: 80% → [MASK]로 교체
        indices_replaced = torch.bernoulli(torch.full(inputs.shape, 0.8)).bool() & masked_indices
        masked_inputs[indices_replaced] = self.tokenizer.mask_token_id
        
        # 전략 2: 남은 것의 50% (전체의 10%) → 랜덤 토큰
        indices_random = torch.bernoulli(torch.full(inputs.shape, 0.5)).bool() & masked_indices & ~indices_replaced
        random_words = torch.randint(self.tokenizer.vocab_size, inputs.shape, dtype=torch.long)
        masked_inputs[indices_random] = random_words[indices_random]
        
        # 전략 3: 나머지 10% → 원본 유지 (아무것도 안 함)
        # 이미 masked_inputs는 원본의 클론이므로 그냥 둠
        
        return masked_inputs, labels


# Data Collator 생성
collator = BertMLMDataCollator(tokenizer, mlm_probability=0.15)
print("✅ BertMLMDataCollator 생성 완료")

In [ ]:
# ============================================================
# 🔍 Step 4: Data Collator 동작 시각화

In [ ]:
# 예시 문장들
sentences = [
    "The cat sat on the mat.",
    "Paris is the capital of France.",
    "Artificial intelligence is transforming the world.",
]

# 토크나이징
encoded = [tokenizer.encode(s, max_length=20, truncation=True) for s in sentences]

# Data Collator 적용
batch = collator(encoded)

print("=" * 60)
print("Data Collator 출력 확인")
print("=" * 60)

for key, value in batch.items():
    print(f"  {key}: shape = {value.shape}")

print()
print("첫 번째 문장 상세 분석:")
print(f"  원본: {sentences[0]}")
print()

# 첫 번째 문장의 마스킹 결과
input_ids_0 = batch['input_ids'][0]
labels_0 = batch['labels'][0]
attn_mask_0 = batch['attention_mask'][0]

print(f"{'위치':<5} {'원본 토큰':<15} {'마스킹 결과':<15} {'정답(labels)':<15} {'상태':<20}")
print("-" * 75)

# 원본 토큰 IDs
orig_ids = encoded[0] + [tokenizer.pad_token_id] * (input_ids_0.size(0) - len(encoded[0]))

for pos in range(min(len(orig_ids), 15)):  # 최대 15개만 표시
    orig_id = orig_ids[pos]
    masked_id = input_ids_0[pos].item()
    label = labels_0[pos].item()
    attn = attn_mask_0[pos].item()
    
    orig_tok = tokenizer.convert_ids_to_tokens([orig_id])[0] if orig_id != 0 else '[PAD]'
    masked_tok = tokenizer.convert_ids_to_tokens([masked_id])[0]
    label_str = tokenizer.convert_ids_to_tokens([label])[0] if label != -100 else '(무시)'
    
    if label == -100:
        status = '마스킹 안 됨'
    elif masked_id == tokenizer.mask_token_id:
        status = '✅ [MASK] 교체'
    elif masked_id != orig_id:
        status = '🔀 랜덤 교체'
    else:
        status = '🔒 원본 유지'
    
    print(f"{pos:<5} {orig_tok:<15} {masked_tok:<15} {label_str:<15} {status:<20}")

In [ ]:
# ============================================================
# 🤗 Step 5: Hugging Face DataCollatorForLanguageModeling 비교

In [ ]:
#
# 우리가 직접 구현한 것과 Hugging Face의 공식 구현을 비교

print("=" * 60)
print("Hugging Face 공식 DataCollator vs 우리 구현 비교")
print("=" * 60)

# Hugging Face 공식 DataCollator
hf_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,              # MLM 사용 (False면 CLM)
    mlm_probability=0.15   # 마스킹 비율 15%
)

# 동일한 데이터에 적용
# HF 형식: {'input_ids': tensor} 형태
hf_examples = [{'input_ids': torch.tensor(e)} for e in encoded]
hf_batch = hf_collator(hf_examples)

print("\nHugging Face DataCollator 출력:")
for key, value in hf_batch.items():
    print(f"  {key}: shape = {value.shape}")

print("\n우리 구현 출력:")
for key, value in batch.items():
    print(f"  {key}: shape = {value.shape}")

print("\n✅ 동일한 구조! 우리 구현이 HF와 동일하게 동작합니다.")

# 마스킹 통계
print("\n마스킹 통계 (우리 구현):")
total_tokens = batch['attention_mask'].sum().item()
masked_tokens = (batch['labels'] != -100).sum().item()
mask_ratio = masked_tokens / total_tokens
print(f"  전체 실제 토큰 수: {total_tokens}")
print(f"  마스킹된 토큰 수: {masked_tokens}")
print(f"  마스킹 비율: {mask_ratio:.1%} (목표: 15%)")

In [ ]:
# ============================================================
# 🚀 Step 6: 실제 학습 루프 맛보기

In [ ]:
#
# 실제 BERT 학습에서 Data Collator가 어떻게 사용되는지 보여줍니다.
# (실제 BERT 모델은 매우 크기 때문에 구조만 설명)

from torch.utils.data import DataLoader, Dataset

class TextDataset(Dataset):
    """
    간단한 텍스트 데이터셋
    """
    def __init__(self, texts, tokenizer, max_length=64):
        self.examples = []
        for text in texts:
            ids = tokenizer.encode(
                text,
                max_length=max_length,
                truncation=True,
                padding=False  # Collator가 패딩 처리
            )
            self.examples.append(ids)
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        return self.examples[idx]


# 예시 텍스트 데이터
texts = [
    "The Eiffel Tower is located in Paris, France.",
    "Machine learning is a subset of artificial intelligence.",
    "Python is a popular programming language for data science.",
    "Natural language processing enables computers to understand human language.",
    "Deep learning models require large amounts of data to train.",
    "The transformer architecture revolutionized natural language processing.",
    "BERT was pre-trained on a large corpus of English text.",
    "Transfer learning allows models to be fine-tuned on specific tasks.",
]

# 데이터셋 생성
dataset = TextDataset(texts, tokenizer)

# DataLoader 생성 (배치 처리 + 마스킹 자동 적용)
dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collator  # 우리가 만든 Data Collator!
)

print("=" * 60)
print("DataLoader + MLM Collator 동작 확인")
print("=" * 60)
print(f"데이터셋 크기: {len(dataset)}개 문장")
print(f"배치 크기: 4")
print(f"배치 수: {len(dataloader)}")
print()

# 첫 번째 배치 확인
first_batch = next(iter(dataloader))
print("첫 번째 배치 구조:")
for key, value in first_batch.items():
    print(f"  {key}: {value.shape}")

# 가상의 학습 루프
print("\n--- 가상의 학습 루프 (구조만 보여줌) ---")
print("""
# 실제 학습 코드 구조:
model = BertForMaskedLM.from_pretrained('bert-base-uncased')
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

for epoch in range(num_epochs):
    for batch in dataloader:      # ← DataCollator가 배치마다 자동 마스킹!
        input_ids = batch['input_ids']
        labels = batch['labels']   # ← -100은 loss 계산 제외
        attention_mask = batch['attention_mask']
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels           # ← 마스킹된 토큰만 loss 계산
        )
        
        loss = outputs.loss         # ← CrossEntropyLoss (ignore_index=-100)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
""")
print("✅ 이 구조로 실제 BERT를 학습시킬 수 있습니다!")

## 📋 정리

### ✅ 이번 실습에서 배운 것

1. **BERT의 핵심 아이디어**
   - 양방향 Transformer 인코더
   - MLM으로 빈칸 채우기 학습

2. **WordPiece 토크나이저**
   - 서브워드 단위로 분리
   - 특수 토큰: [CLS], [SEP], [MASK], [PAD]

3. **MLM 마스킹 규칙 (15%)**
   - 80% → [MASK]
   - 10% → 랜덤 토큰
   - 10% → 원본 유지

4. **Data Collator 구현**
   - 배치 패딩 처리
   - 자동 마스킹
   - labels=-100으로 무시 위치 표시

### ➡️ 다음 실습 (068)
- **MLM 마스킹 심화 구현**
- 다양한 마스킹 전략 비교
- Whole Word Masking 구현